# vidcap demo

HDMI in → filter → timestamp → HDMI out on the PYNQ-Z2. Run the cells top to bottom with an HDMI source on the
input connector (1280x720 @ 60 Hz) and a monitor on the output. Without a source the board shows a colour-bar
test pattern after 5 seconds.

Copy this folder to `/home/xilinx/jupyter_notebooks/vidcap/` (the `make deploy` target does it).

In [ ]:
import sys, time
sys.path.insert(0, '.')            # the 'vidcap' driver package sits next to this notebook
from vidcap import Vidcap

v = Vidcap.on_pynq('vidcap.bit')   # downloads the bitstream and checks the board answers
v.start()                          # starts the DMA engines, shows live video, sets the clock from this board
v.status()

## What is the input doing?
`hdmi_good` means locked, 1280x720, frames arriving. `fallback_active` means the test pattern has taken over.

In [ ]:
v.input.status()

## Filters
The mode changes on a frame boundary, so there is no tearing.

In [ ]:
for mode in ['passthrough', 'invert', 'grayscale', 'threshold', 'sobel', 'blur', 'sharpen']:
    v.set_filter(mode, threshold=110)
    print(mode)
    time.sleep(2)
v.set_filter('passthrough')

## Test pattern
Colour bars with a moving box: if the box stops, the pipeline has stalled.

In [ ]:
v.use_test_pattern(True);  time.sleep(3)
v.use_test_pattern(False)

## Snapshot
Captures the next complete frame from the *selected input*, before filtering.

In [ ]:
img = v.snapshot()             # (720, 1280, 3) uint8, B,G,R order
from PIL import Image
Image.fromarray(img[..., ::-1]).resize((640, 360))

## Video memory
Show any 1280x720 image instead of live video (it goes through the filters and gets the timestamp).

In [ ]:
import numpy as np
gradient = np.zeros((720, 1280, 3), np.uint8)
gradient[..., 0] = np.linspace(0, 255, 1280, dtype=np.uint8)           # blue ramp
gradient[..., 2] = np.linspace(0, 255, 720, dtype=np.uint8)[:, None]   # red ramp
v.show_image(gradient)
time.sleep(3)
v.show_image(img)                 # the captured snapshot, shown back
time.sleep(3)
v.show_live()

## Timestamp
The on-screen time is a hardware clock that was set from this board's clock at `start()`.

In [ ]:
v.overlay.configure(scale=3, margin=24, fg=(255, 255, 0), bg=(0, 0, 0))
print('hardware clock:', v.rtc.get(), ' ahead of the board clock by', round(v.rtc.drift_against(), 3), 's')

In [ ]:
# to correct drift: v.rtc.cycles_per_second(100_000_000 + adjustment), then v.sync_time()
v.overlay.enable(False)

## Diagnostics
VDMA error lists should be empty while video is flowing.

In [ ]:
v.status()